In [2]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('../../../')

In [3]:
import pandas as pd
from statsmodels.formula import api as smf
from data_helpers.data_loaders import PandasDataLoader 
from data_helpers.data_prep import dynamic_treatments
import numpy as np
from tqdm.auto import tqdm


## Prepare and Filter Dataset


In [4]:
model_name = 'emh'
sample_splits_df = pd.read_feather('../../../data/preprocessed/train_test_split.ft')
time_aggregated_dataset = pd.read_feather('../../../data/preprocessed/time_aggregate_dataset.ft')
time_aggregated_dataset = time_aggregated_dataset[~time_aggregated_dataset.treatment.isin(dynamic_treatments)]
time_aggregated_dataset = time_aggregated_dataset.query('round <= 5 and time <= 120')

## Keep relevant columns and prepare train-test loader

In [5]:
key_columns = ['treatment', 'game', 'round', 'time', 'n_unique_deals_round']
rounds = range(1,5)
n_deal_prices = range(0,6)
pdl = PandasDataLoader(sample_splits_df, time_aggregated_dataset)

## Fit and evaluate models

In [6]:
np.random.seed(1)
all_results = []
for i in tqdm(range(pdl.max_samples)):
    train_df, test_df = pdl.get_sample_split_dataset(i)

    sub_test_set = test_df.query('round < 5 and n_unique_deals_round < 6').copy()
    prediction = 1.0
    denom = sub_test_set['allocative_efficiency_round'].copy()
    denom[denom==0] = 1
    denom = denom.values

    result_test_df = sub_test_set[key_columns].copy()            
    result_test_df.loc[:, 'ae_ape'] = (np.abs(prediction - sub_test_set['allocative_efficiency_round'].values)/denom)
    result_test_df.loc[:, 'target'] = sub_test_set['allocative_efficiency_round'].values
    

    result_test_df.loc[:, 'sample_id'] = i
    all_results.append(result_test_df)

  0%|          | 0/50 [00:00<?, ?it/s]

## Combine Results

In [7]:
all_results_df = pd.concat(all_results, ignore_index = True)
all_results_df['model'] = model_name

## Persist Performance Results

In [8]:
all_results_df.to_feather('../../../data/results/allocative_efficiency/'+model_name+'.ft')